In [24]:
##### SI Table 1: metrics of RF model performance

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score

In [25]:
##### SET-UP

# Get the current working directory
cd = Path.cwd().parent

# import reference data 
ref_data = pd.read_csv(f"{cd}/Data/subnational_intensities.csv")

# import model predictions
capital_rf = pd.read_parquet(f'{cd}/Results/RF_models/capital_rf_spatial_CV/predictions.parquet')
labor_rf = pd.read_parquet(f'{cd}/Results/RF_models/labor_rf_spatial_CV/predictions.parquet')

capital_qrf = pd.read_parquet(f'{cd}/Results/RF_models/capital_qrf_spatial_CV/predictions.parquet')
labor_qrf = pd.read_parquet(f'{cd}/Results/RF_models/labor_qrf_spatial_CV/predictions.parquet')

# country groups 
country_codes = pd.read_csv(f"{cd}/Data/country_names.csv", encoding="cp1252")

In [26]:
##### PREP DATA

# Add original intensity data 
capital_rf = capital_rf.merge(ref_data, on=['PROJ_ID', 'country_ID'], how='left')
labor_rf = labor_rf.merge(ref_data, on=['PROJ_ID', 'country_ID'], how='left')
capital_qrf = capital_qrf.merge(ref_data, on=['PROJ_ID', 'country_ID'], how='left')
labor_qrf = labor_qrf.merge(ref_data, on=['PROJ_ID', 'country_ID'], how='left')

# Estimate actual predicted intensity (not relative)
capital_rf['predicted_log_capital_intensity_USD_per_million_tonne'] = capital_rf['prediction'] + capital_rf['log_country_capital_intensity_USD_per_million_tonne']
capital_rf['predicted_capital_intensity_USD_per_million_tonne'] = np.expm1(capital_rf['predicted_log_capital_intensity_USD_per_million_tonne'])
capital_qrf['q10_predicted_log_capital_intensity_USD_per_million_tonne'] = capital_qrf['q10'] + capital_qrf['log_country_capital_intensity_USD_per_million_tonne']
capital_qrf['q90_predicted_log_capital_intensity_USD_per_million_tonne'] = capital_qrf['q90'] + capital_qrf['log_country_capital_intensity_USD_per_million_tonne']

labor_rf['predicted_log_labor_intensity_jobs_per_million_tonne'] = labor_rf['prediction'] + labor_rf['log_country_labor_intensity_jobs_per_million_tonne']
labor_rf['predicted_labor_intensity_jobs_per_million_tonne'] = np.expm1(labor_rf['predicted_log_labor_intensity_jobs_per_million_tonne'])
labor_qrf['q10_predicted_log_labor_intensity_jobs_per_million_tonne'] = labor_qrf['q10'] + labor_qrf['log_country_labor_intensity_jobs_per_million_tonne']
labor_qrf['q90_predicted_log_labor_intensity_jobs_per_million_tonne'] = labor_qrf['q90'] + labor_qrf['log_country_labor_intensity_jobs_per_million_tonne']

In [27]:
##### DEFINE FUNCTIONS

def compute_fold_metrics(df, target_col, pred_col):
    rows = []
    for fold, g in df.groupby("fold"):
        y_true = g[target_col]
        y_pred = g[pred_col]
        resid = y_true - y_pred
        rows.append({
            "fold": fold,
            "R2": r2_score(y_true, y_pred),
            "PBIAS": resid.mean() / y_true.mean() * 100 if y_true.mean() != 0 else np.nan,
        })
    return pd.DataFrame(rows)

def compute_fold_coverage(df, target_col, q10_col, q90_col):
    rows = []
    for fold, g in df.groupby("fold"):
        y_true = g[target_col]
        low = g[q10_col]
        high = g[q90_col]
        covered = (y_true >= low) & (y_true <= high)
        rows.append({"fold": fold, "coverage": covered.mean()})
    return pd.DataFrame(rows)

def summarize_spec(df, target_col, pred_col, qrf_df=None, q10_col=None, q90_col=None, report_pbias=True):
    fold_metrics = compute_fold_metrics(df, target_col, pred_col)
    result = {
        "R2": fold_metrics["R2"].mean(),
        "VAR_R2": fold_metrics["R2"].var(),
        "PBIAS": fold_metrics["PBIAS"].mean() if report_pbias else np.nan,
        "VAR_PBIAS": fold_metrics["PBIAS"].var() if report_pbias else np.nan,
    }
    if qrf_df is not None:
        fold_cov = compute_fold_coverage(qrf_df, target_col, q10_col, q90_col)
        result["QRF_coverage"] = fold_cov["coverage"].mean()
    else:
        result["QRF_coverage"] = np.nan
    return result

In [28]:
##### CALCULATE METRICS

# use test splits only
capital_test_rf  = capital_rf[capital_rf["split"] == "test"]
labor_test_rf    = labor_rf[labor_rf["split"] == "test"]
capital_test_qrf = capital_qrf[capital_qrf["split"] == "test"]
labor_test_qrf   = labor_qrf[labor_qrf["split"] == "test"]

rows = []

# capital
rows.append({
    "Model": "Capital intensity", "Specification": "Relative",
    **summarize_spec(
        capital_test_rf, "rtv_log_capital_intensity_USD_per_million_tonne", "prediction",
        qrf_df=capital_test_qrf, q10_col="q10", q90_col="q90",
        report_pbias=False
    )
})
rows.append({
    "Model": "Capital intensity", "Specification": "Absolute",
    **summarize_spec(
        capital_test_rf,
        "log_region_capital_intensity_USD_per_million_tonne",
        "predicted_log_capital_intensity_USD_per_million_tonne",
        qrf_df=capital_test_qrf,
        q10_col="q10_predicted_log_capital_intensity_USD_per_million_tonne",
        q90_col="q90_predicted_log_capital_intensity_USD_per_million_tonne"
    )
})
rows.append({
    "Model": "Capital intensity", "Specification": "Null model",
    **summarize_spec(
        capital_test_rf,
        "log_region_capital_intensity_USD_per_million_tonne",
        "log_country_capital_intensity_USD_per_million_tonne"
    )
})

# labor
rows.append({
    "Model": "Labor intensity", "Specification": "Relative",
    **summarize_spec(
        labor_test_rf, "rtv_log_labor_intensity_jobs_per_million_tonne", "prediction",
        qrf_df=labor_test_qrf, q10_col="q10", q90_col="q90",
        report_pbias=False
    )
})
rows.append({
    "Model": "Labor intensity", "Specification": "Absolute",
    **summarize_spec(
        labor_test_rf,
        "log_region_labor_intensity_jobs_per_million_tonne",
        "predicted_log_labor_intensity_jobs_per_million_tonne",
        qrf_df=labor_test_qrf,
        q10_col="q10_predicted_log_labor_intensity_jobs_per_million_tonne",
        q90_col="q90_predicted_log_labor_intensity_jobs_per_million_tonne"
    )
})
rows.append({
    "Model": "Labor intensity", "Specification": "Null model",
    **summarize_spec(
        labor_test_rf,
        "log_region_labor_intensity_jobs_per_million_tonne",
        "log_country_labor_intensity_jobs_per_million_tonne"
    )
})

summary_full = pd.DataFrame(rows)[
    ["Model", "Specification", "R2", "VAR_R2", "PBIAS", "VAR_PBIAS", "QRF_coverage"]
]

# print
print("\n── FULL MODEL EVALUATION SUMMARY ─────────────────────────────────────────")
print(summary_full.round(3).to_string(index=False))


── FULL MODEL EVALUATION SUMMARY ─────────────────────────────────────────
            Model Specification     R2  VAR_R2  PBIAS  VAR_PBIAS  QRF_coverage
Capital intensity      Relative  0.248   0.072    NaN        NaN         0.747
Capital intensity      Absolute  0.378   0.003 -0.082      1.685         0.747
Capital intensity    Null model -0.004   0.088  1.529      2.111           NaN
  Labor intensity      Relative  0.521   0.046    NaN        NaN         0.819
  Labor intensity      Absolute  0.748   0.004 -0.434      3.906         0.819
  Labor intensity    Null model  0.294   0.176  3.099      7.838           NaN


In [29]:
### Table of performance metrics by country and income group 
# pooled R2

MIN_N = 5  # minimum test observations for a group to be reported

INCOME_ORDER = ["Low income", "Lower middle income", "Upper middle income", "High income"]
SPEC_ORDER   = ["Null model", "Absolute", "Relative"]
MODEL_ORDER  = ["Capital intensity", "Labor intensity"]

# column groups
def group_r2(df, group_col, target_col, pred_col, min_n=MIN_N):
    rows = []
    for key, g in df.groupby(group_col):
        if len(g) < min_n:
            continue
        rows.append({group_col: key, "n": len(g),
                     "R2": r2_score(g[target_col], g[pred_col])})
    return pd.DataFrame(rows, columns=[group_col, "n", "R2"])

# add income groups 
n_before = (len(capital_test_rf), len(labor_test_rf))
capital_test_rf = capital_test_rf.merge(country_codes, left_on="country_ID", right_on="ISO3", how="left")
labor_test_rf   = labor_test_rf.merge(country_codes, left_on="country_ID", right_on="ISO3", how="left")
assert (len(capital_test_rf), len(labor_test_rf)) == n_before, \
    "Merge changed row count: country_codes has duplicate ISO3 rows"

for name, d in [("capital", capital_test_rf), ("labor", labor_test_rf)]:
    missing = d.loc[d["income_group"].isna(), "country_ID"].unique()
    if len(missing):
        print(f"WARNING: {name}: no income group for country_ID(s): {list(missing)}")

# specify each model
specs = [
    ("Capital intensity", "Relative",   capital_test_rf,
     "rtv_log_capital_intensity_USD_per_million_tonne", "prediction"),
    ("Capital intensity", "Absolute",   capital_test_rf,
     "log_region_capital_intensity_USD_per_million_tonne",
     "predicted_log_capital_intensity_USD_per_million_tonne"),
    ("Capital intensity", "Null model", capital_test_rf,
     "log_region_capital_intensity_USD_per_million_tonne",
     "log_country_capital_intensity_USD_per_million_tonne"),
    ("Labor intensity", "Relative",     labor_test_rf,
     "rtv_log_labor_intensity_jobs_per_million_tonne", "prediction"),
    ("Labor intensity", "Absolute",     labor_test_rf,
     "log_region_labor_intensity_jobs_per_million_tonne",
     "predicted_log_labor_intensity_jobs_per_million_tonne"),
    ("Labor intensity", "Null model",   labor_test_rf,
     "log_region_labor_intensity_jobs_per_million_tonne",
     "log_country_labor_intensity_jobs_per_million_tonne"),
]

def summarize_by_group(group_col):
    out = []
    for model, spec, df, target, pred in specs:
        m = group_r2(df, group_col, target, pred)
        m.insert(0, "Specification", spec)
        m.insert(0, "Model", model)
        out.append(m)
    return pd.concat(out, ignore_index=True)

def n_label(n_by_model):
    """n_by_model: Series indexed by Model. One value if equal across models, else 'cap/lab'."""
    vals = [n_by_model.get(m) for m in MODEL_ORDER]
    vals = [int(v) for v in vals if pd.notna(v)]
    if len(set(vals)) <= 1:
        return f"n={vals[0]}" if vals else "n=–"
    return "n=" + "/".join(str(v) for v in vals)  

# Income group table
def income_table(summary, group_col="income_group"):
    r2 = summary.pivot(index=["Model", "Specification"], columns=group_col, values="R2")
    n  = summary.groupby([group_col, "Model"])["n"].first().unstack("Model")

    groups = [g for g in INCOME_ORDER if g in r2.columns] + \
             [g for g in r2.columns if g not in INCOME_ORDER]      
    rows = [(m, s) for m in MODEL_ORDER for s in SPEC_ORDER if (m, s) in r2.index]

    r2 = r2.loc[rows, groups]
    r2.columns = [f"{g} ({n_label(n.loc[g])})" for g in groups]
    return r2

summary_income = summarize_by_group("income_group")
tbl_income = income_table(summary_income)

# Country table
def country_table(summary, group_col="country_ID"):
    r2 = summary.pivot(index=group_col, columns=["Model", "Specification"], values="R2")
    n  = summary.groupby([group_col, "Model"])["n"].first().unstack("Model")

    cols = [(m, s) for m in MODEL_ORDER for s in SPEC_ORDER if (m, s) in r2.columns]
    r2 = r2[cols]
    r2.columns = [f"{m.split()[0]}-{s}" for m, s in cols]            

    r2.index = [f"{c} ({n_label(n.loc[c])})" for c in r2.index]
    r2.index.name = "Country"
    return r2.sort_index()

summary_country = summarize_by_group("country_ID")
tbl_country = country_table(summary_country)

# ── output ────────────────────────────────────────────────────────────────────
print("\n── R² BY INCOME GROUP ──────────────────────────────────")
print(tbl_income.round(2).to_string())
print("\n── R² BY COUNTRY ───────────────────────────────────────")
print(tbl_country.round(2).to_string())


── R² BY INCOME GROUP (out-of-sample) ──────────────────────────────────
                                 Low income (n=14)  Lower middle income (n=50/638)  Upper middle income (n=1249/1330)  High income (n=1450/1599)
Model             Specification                                                                                                                 
Capital intensity Null model                   NaN                            0.03                               0.05                       0.16
                  Absolute                     NaN                            0.01                               0.49                       0.52
                  Relative                     NaN                           -0.13                               0.39                       0.35
Labor intensity   Null model                 -0.16                           -0.02                              -0.02                       0.24
                  Absolute                   -0.09      